# iconflow: research reproduction and extension

The code below is generated directly from the repository packages. Unit tests check implementation behavior, not reproduced paper benchmarks. Runs are illustrative and must be validated before claims of improvement.

In [ ]:
import torch
print("torch:",torch.__version__,"CUDA:",torch.cuda.is_available())
!pip -q install Pillow numpy pytest matplotlib opencv-python-headless

In [ ]:
!mkdir -p iconflow tests

In [ ]:
%%writefile iconflow/__init__.py
"""Iconflow research prototype package."""


In [ ]:
%%writefile iconflow/train.py
"""Conditional flow matching from scratch: straight-line velocity field + Euler ODE sampler.
ICLR 2023 paper-inspired SMALL pixel-space model trained on procedural icons.
python train.py --steps 1000 --batch-size 32  (Colab T4 recommended)
"""
import argparse,math,csv,random
from pathlib import Path
import numpy as np
from PIL import Image,ImageDraw
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import Dataset,DataLoader

CLASSES=['circle','star','lightning','chat','diamond']
COLORS=['#4C63D2','#F05D66','#1FA89A','#F5A623','#9D52C7']


def make_icon(index,size=24,seed=43):
    r=random.Random(seed+index*7349);cls=index%len(CLASSES);ss=size*3
    bg=(r.randrange(235,255),r.randrange(235,255),r.randrange(235,255))
    im=Image.new('RGB',(ss,ss),bg);d=ImageDraw.Draw(im)
    color=r.choice(COLORS)
    cx=(.5+r.uniform(-.10,.10))*ss;cy=(.5+r.uniform(-.10,.10))*ss
    a=r.uniform(.24,.36)*ss;x0,y0,x1,y1=(cx-a,cy-a,cx+a,cy+a)
    if cls==0:d.ellipse((x0,y0,x1,y1),fill=color)
    elif cls==1:
        outer=[]
        for k in range(10):
            ang=-math.pi/2+k*math.pi/5; rad=a if k%2==0 else a*.44
            outer.append((cx+math.cos(ang)*rad,cy+math.sin(ang)*rad))
        d.polygon(outer,fill=color)
    elif cls==2:
        pts=[(cx+.05*a,cy-a),(cx-.8*a,cy+.15*a),(cx-.12*a,cy+.15*a),(cx-.3*a,cy+a),(cx+.75*a,cy-.24*a),(cx+.12*a,cy-.24*a)]
        d.polygon(pts,fill=color)
    elif cls==3:
        d.rounded_rectangle((x0,y0,x1,y1-.1*a),radius=.25*a,fill=color)
        d.polygon([(cx-.42*a,cy+.7*a),(cx-.42*a,cy+1.1*a),(cx+.03*a,cy+.55*a)],fill=color)
        for dx in (-.4,0,.4):d.ellipse((cx+dx*a-.08*a,cy-.08*a,cx+dx*a+.08*a,cy+.08*a),fill='white')
    else:d.polygon([(cx,cy-a),(cx+a,cy),(cx,cy+a),(cx-a,cy)],fill=color)
    return im.resize((size,size),resample=Image.Resampling.LANCZOS),cls


class Icons(Dataset):
    def __init__(self,size=24,count=3000,seed=43):self.size=size;self.count=count;self.seed=seed
    def __len__(self):return self.count
    def __getitem__(self,idx):
        image,cls=make_icon(idx,self.size,self.seed)
        x=torch.from_numpy(np.asarray(image,dtype=np.float32).copy()).permute(2,0,1)/127.5-1
        return x,cls


class TimeEmbed(nn.Module):
    def __init__(self,dim):
        super().__init__();self.dim=dim
        self.mlp=nn.Sequential(nn.Linear(dim,dim*4),nn.SiLU(),nn.Linear(dim*4,dim))
    def forward(self,t):
        half=self.dim//2
        f=torch.exp(-math.log(10000)*torch.arange(half,device=t.device,dtype=t.dtype)/max(half-1,1))
        z=t[:,None]*f[None,:]*1000
        embedding=torch.cat([z.sin(),z.cos()],dim=-1)
        return self.mlp(embedding)


class ResBlock(nn.Module):
    def __init__(self,inch,outch,edim):
        super().__init__()
        self.norm1=nn.GroupNorm(8,inch);self.conv1=nn.Conv2d(inch,outch,3,padding=1)
        self.norm2=nn.GroupNorm(8,outch);self.conv2=nn.Conv2d(outch,outch,3,padding=1)
        self.emb=nn.Linear(edim,outch);self.shortcut=nn.Conv2d(inch,outch,1) if inch!=outch else nn.Identity()
    def forward(self,x,e):
        z=self.conv1(F.silu(self.norm1(x)))
        z=z+self.emb(F.silu(e))[:,:,None,None]
        return self.shortcut(x)+self.conv2(F.silu(self.norm2(z)))


class VelocityUNet(nn.Module):
    """Tiny class-conditioned U-Net, built without pretrained diffusion/flow libraries."""
    def __init__(self,base=32,num_classes=5):
        super().__init__()
        ed=base*4
        self.time=TimeEmbed(ed);self.num_classes=num_classes;self.label=nn.Embedding(num_classes+1,ed)  # final index is CFG null token
        self.inconv=nn.Conv2d(3,base,3,padding=1)
        self.enc1=ResBlock(base,base,ed)
        self.down1=nn.Conv2d(base,base*2,3,stride=2,padding=1)
        self.enc2=ResBlock(base*2,base*2,ed)
        self.down2=nn.Conv2d(base*2,base*4,3,stride=2,padding=1)
        self.mid=ResBlock(base*4,base*4,ed)
        self.dec2=ResBlock(base*6,base*2,ed)
        self.dec1=ResBlock(base*3,base,ed)
        self.outnorm=nn.GroupNorm(8,base);self.outconv=nn.Conv2d(base,3,3,padding=1)
        nn.init.zeros_(self.outconv.weight);nn.init.zeros_(self.outconv.bias)
    def forward(self,x,t,c):
        if c.dtype != torch.long or bool(((c<0)|(c>self.num_classes)).any()):
            raise ValueError('Labels must be LongTensor in [0, num_classes], with num_classes as null token')
        e=self.time(t)+self.label(c)
        x1=self.enc1(self.inconv(x),e)
        x2=self.enc2(self.down1(x1),e)
        z=self.mid(self.down2(x2),e)
        z=F.interpolate(z,size=x2.shape[-2:],mode='nearest')
        z=self.dec2(torch.cat([z,x2],dim=1),e)
        z=F.interpolate(z,size=x1.shape[-2:],mode='nearest')
        z=self.dec1(torch.cat([z,x1],dim=1),e)
        return self.outconv(F.silu(self.outnorm(z)))


def flow_interpolant(x0,x1,t,sigma_min=.01):
    """Gaussian OT conditional path and velocity (Lipman et al., ICLR 2023).

    x_t=(1-(1-sigma_min)t)*x0+t*x1
    u_t=x1-(1-sigma_min)*x0. sigma_min>0 avoids a degenerate final
    conditional distribution. sigma_min=0 is the straight-line limit.
    """
    if not (0<=sigma_min<1): raise ValueError('sigma_min must be in [0,1)')
    t=t.reshape(-1,1,1,1)
    xt=(1-(1-sigma_min)*t)*x0+t*x1
    target=x1-(1-sigma_min)*x0
    return xt,target


def conditional_flow_loss(model,x1,labels,sigma_min=.01,noise=None,times=None,cfg_dropout=0.):
    """FM objective. With cfg_dropout>0 this learns an unconditional null-token branch.

    CFG is an independent extension, not a requirement of the Lipman et al. objective.
    """
    if not 0. <= cfg_dropout <= 1.:
        raise ValueError('cfg_dropout must lie in [0,1]')
    x0=torch.randn_like(x1) if noise is None else noise
    t=torch.rand(len(x1),device=x1.device) if times is None else times
    xt,velocity=flow_interpolant(x0,x1,t,sigma_min)
    if cfg_dropout:
        mask=torch.rand(len(labels),device=labels.device)<cfg_dropout
        labels=torch.where(mask,torch.full_like(labels,model.num_classes),labels)
    prediction=model(xt,t,labels)
    return F.mse_loss(prediction,velocity)


@torch.no_grad()
def sample(model,labels,size=24,steps=30,device='cpu',solver='euler',
           noise=None,clip_output=True,guidance_scale=1.):
    """ODE dx/dt=v_theta(x,t); paper-faithful Euler plus Heun EXTENSION.

    Euler evaluates velocity at LEFT time i/steps, not (i+.5)/steps.
    Heun (predictor-corrector) uses an additional function evaluation.
    Clipping is only for final image rendering, never during ODE steps.
    """
    if solver not in ('euler','heun'): raise ValueError('solver must be euler/heun')
    if steps<1:raise ValueError('steps must be positive')
    if guidance_scale < 0:raise ValueError('guidance_scale must be nonnegative')
    model.eval()
    labels=labels.to(device=device,dtype=torch.long)
    if noise is not None and tuple(noise.shape)!=(len(labels),3,size,size):
        raise ValueError('Noise must have shape (len(labels),3,size,size)')
    x=(torch.randn((len(labels),3,size,size),device=device) if noise is None
       else noise.to(device).clone())
    dt=1/steps
    def guided_velocity(x,t):
        conditioned=model(x,t,labels)
        if guidance_scale==1.:
            return conditioned
        unconditioned=model(x,t,torch.full_like(labels,model.num_classes))
        return unconditioned+guidance_scale*(conditioned-unconditioned)
    for i in range(steps):
        t=torch.full((len(labels),),i/steps,device=device)
        v=guided_velocity(x,t)
        if solver=='heun':
            next_t=torch.full((len(labels),),(i+1)/steps,device=device)
            v_next=guided_velocity(x+dt*v,next_t)
            x=x+.5*dt*(v+v_next)
        else:
            x=x+dt*v
    return x.clamp(-1,1) if clip_output else x

def to_pil(x):
    a=((x.detach().cpu().permute(1,2,0).clamp(-1,1)+1)*127.5).numpy().astype('uint8')
    return Image.fromarray(a)


def save_grid(images,path,size=24,cols=5,scale=6):
    count=images.shape[0];rows=math.ceil(count/cols)
    grid=Image.new('RGB',(cols*size*scale,rows*size*scale),'white')
    for n,arr in enumerate(images):
        img=to_pil(arr).resize((size*scale,size*scale),Image.Resampling.NEAREST)
        grid.paste(img,((n%cols)*size*scale,(n//cols)*size*scale))
    grid.save(path)


def run(args):
    torch.manual_seed(args.seed);random.seed(args.seed);np.random.seed(args.seed)
    device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    if args.size%4!=0:raise ValueError('--size must be divisible by 4')
    out=Path(args.out);out.mkdir(parents=True,exist_ok=True)
    saved=torch.load(args.checkpoint,map_location=device,weights_only=True) if args.checkpoint else None
    effective_width=saved.get('width',args.width) if isinstance(saved,dict) else args.width
    effective_size=saved.get('size',args.size) if isinstance(saved,dict) else args.size
    if args.sample_only and args.guidance_scale != 1 and (not isinstance(saved,dict) or saved.get('cfg_dropout',0.)<=0):
        raise ValueError('Classifier-free guidance requires checkpoint trained with --cfg-dropout > 0')
    model=VelocityUNet(base=effective_width).to(device)
    if saved is not None:
        model.load_state_dict(saved['model'] if 'model' in saved else saved)
    labels=torch.tensor([i%5 for i in range(20)],device=device)
    if args.sample_only:
        save_grid(sample(model,labels,size=effective_size,steps=args.sample_steps,device=device,solver=args.solver,guidance_scale=args.guidance_scale),out/'generated_icons.png',effective_size)
        print('Wrote',out/'generated_icons.png')
        if args.export_svg:
            from iconflow.vectorize import vectorize_grid
            info=vectorize_grid(Image.open(out/'generated_icons.png'),out/'editable_svg_icons',effective_size)
            print('Exported',len(info),'editable SVG candidates')
        return
    ds=Icons(size=args.size,count=args.samples,seed=args.seed)
    dl=DataLoader(ds,batch_size=args.batch_size,shuffle=True,num_workers=0)
    optim=torch.optim.AdamW(model.parameters(),lr=args.lr,weight_decay=1e-4)
    itr=iter(dl);history=[]
    for step in range(1,args.steps+1):
        try:x,c=next(itr)
        except StopIteration:itr=iter(dl);x,c=next(itr)
        x=x.to(device);c=c.to(device);model.train()
        loss=conditional_flow_loss(model,x,c,sigma_min=args.sigma_min,cfg_dropout=args.cfg_dropout)
        optim.zero_grad();loss.backward();nn.utils.clip_grad_norm_(model.parameters(),1.);optim.step()
        history.append([step,float(loss.detach())])
        if step==1 or step%max(1,args.steps//10)==0:
            print(f'step {step}/{args.steps} flow loss {float(loss.detach()):.4f}')
    torch.save({'model':model.state_dict(),'size':args.size,'width':args.width,'steps':args.steps,'sigma_min':args.sigma_min,'cfg_dropout':args.cfg_dropout},out/'flow_model.pt')
    with (out/'training.csv').open('w',newline='') as f:
        w=csv.writer(f);w.writerow(['step','flow_mse']);w.writerows(history)
    # Same evaluation seeds across runs, generate one independent image per category.
    torch.manual_seed(args.seed+999)
    if args.guidance_scale!=1 and args.cfg_dropout<=0:
        raise ValueError('--guidance-scale !=1 requires --cfg-dropout > 0 at training')
    generated=sample(model,labels,size=args.size,steps=args.sample_steps,device=device,solver=args.solver,guidance_scale=args.guidance_scale)
    save_grid(generated,out/'generated_icons.png',args.size)
    if args.export_svg:
        from iconflow.vectorize import vectorize_grid
        info=vectorize_grid(Image.open(out/'generated_icons.png'),out/'editable_svg_icons',args.size)
        print('Exported',len(info),'editable SVG candidates')
    reference=torch.stack([ds[i][0] for i in range(20)])
    save_grid(reference,out/'training_examples.png',args.size)
    import json
    (out/'metrics.json').write_text(json.dumps({'loss_initial':history[0][1],'loss_final':history[-1][1],
        'loss_last_50_mean':float(np.mean([v for _,v in history[-50:]])),
        'classes':CLASSES,'cfg_dropout':args.cfg_dropout,'guidance_scale':args.guidance_scale,'solver':args.solver,'evaluation_note':'Flow matching MSE is not an image-quality metric; visually inspect generated grid and extend with FID on a held-out image dataset.'},indent=2))
    print('Saved flow_model.pt, generated_icons.png, training_examples.png, training.csv, metrics.json to',out,'device',device)


def parse_args(argv=None):
    p=argparse.ArgumentParser();p.add_argument('--steps',type=int,default=1000)
    p.add_argument('--batch-size',type=int,default=32);p.add_argument('--samples',type=int,default=3000)
    p.add_argument('--size',type=int,default=24);p.add_argument('--width',type=int,default=32)
    p.add_argument('--lr',type=float,default=2e-4);p.add_argument('--sample-steps',type=int,default=30)
    p.add_argument('--seed',type=int,default=42);p.add_argument('--out',default='outputs')
    p.add_argument('--checkpoint',default='');p.add_argument('--sample-only',action='store_true')
    p.add_argument('--sigma-min',type=float,default=.01,help='Gaussian OT conditional path endpoint variance; 0 is straight-line limit')
    p.add_argument('--solver',choices=['euler','heun'],default='euler',help='Heun is an optional sampling extension')
    p.add_argument('--export-svg',action='store_true',help='Contour-trace sampled images into editable SVGs (requires opencv)')
    p.add_argument('--cfg-dropout',type=float,default=0.0,help='EXTENSION: fraction of training labels replaced by unconditional null token (e.g. 0.15)')
    p.add_argument('--guidance-scale',type=float,default=1.0,help='EXTENSION: CFG generation scale; 1.0 reproduces base conditional velocity')
    return p.parse_args(argv)

if __name__=='__main__':run(parse_args())


In [ ]:
%%writefile iconflow/vectorize.py
"""Real editable SVG contours from pixel-space generated icon PNGs.

Quantize the source image and trace simplified contours. Output contains vector
<path> elements, no PNG/data URI. Requires opencv-python-headless and Pillow.
Not vector diffusion; this is a classical, separately attributed postprocess.
"""
import argparse
from pathlib import Path
import numpy as np
from PIL import Image,ImageFilter


def raster_to_svg(image,output,colors=4,simplify=.012,blur=.3,foreground=None):
    import cv2
    im=image.convert('RGB')
    im=im.filter(ImageFilter.GaussianBlur(radius=max(0,float(blur))))
    rgb=np.asarray(im,dtype=np.uint8)
    h,w=rgb.shape[:2]
    # Median border color is the canvas background estimate.
    edge=np.concatenate([rgb[0,:,:],rgb[-1,:,:],rgb[:,0,:],rgb[:,-1,:]],axis=0)
    bg=np.median(edge,axis=0).astype(np.uint8)
    samples=rgb.reshape(-1,3).astype(np.float32)
    criteria=(cv2.TERM_CRITERIA_EPS+cv2.TERM_CRITERIA_MAX_ITER,25,.25)
    cv2.setRNGSeed(1)
    _,idx,centers=cv2.kmeans(samples,max(2,int(colors)),None,criteria,4,cv2.KMEANS_PP_CENTERS)
    centers=np.clip(centers,0,255).astype(np.uint8)
    group=idx.reshape(h,w)
    # Majority border cluster designated background; remaining regions become paths.
    border_labels=np.concatenate([group[0,:],group[-1,:],group[:,0],group[:,-1]])
    bg_label=int(np.bincount(border_labels,minlength=len(centers)).argmax())
    fill_bg='#%02x%02x%02x'%tuple(int(v) for v in centers[bg_label])
    lines=[f'<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 {w} {h}" width="{w*8}" height="{h*8}">',
           '<title>Editable vector traces from a generated raster icon</title>',
           f'<rect id="editable-background" width="{w}" height="{h}" fill="{fill_bg}"/>']
    paths=0
    for lab in range(len(centers)):
        if lab==bg_label:continue
        mask=(group==lab).astype(np.uint8)*255
        contours,_=cv2.findContours(mask,cv2.RETR_EXTERNAL,cv2.CHAIN_APPROX_SIMPLE)
        color=(foreground if foreground else '#%02x%02x%02x'%tuple(int(v) for v in centers[lab]))
        for contour in contours:
            if cv2.contourArea(contour)<max(3,w*h*.003):continue
            approx=cv2.approxPolyDP(contour,max(.15,float(simplify)*cv2.arcLength(contour,True)),True).reshape(-1,2)
            if len(approx)<3:continue
            d='M '+' L '.join(f'{int(x)} {int(y)}' for x,y in approx)+' Z'
            lines.append(f'<path id="vector-shape-{paths+1}" d="{d}" fill="{color}"/>')
            paths+=1
    lines.append('</svg>')
    Path(output).write_text('\n'.join(lines),encoding='utf-8')
    return {'width':w,'height':h,'paths':paths,'background':fill_bg,'output':str(output)}


def vectorize_grid(image,output_dir,icon_size,cols=5,count=20,scale=6):
    """Slice nearest-neighbor tiled grid and SVG-trace each upscaled tile."""
    grid=image.convert('RGB');out=Path(output_dir);out.mkdir(parents=True,exist_ok=True)
    info=[]
    for i in range(count):
        x=(i%cols)*icon_size*scale;y=(i//cols)*icon_size*scale
        tile=grid.crop((x,y,x+icon_size*scale,y+icon_size*scale))
        # Match original low-res structure before contour extraction.
        tile=tile.resize((icon_size,icon_size),Image.Resampling.LANCZOS)
        dest=out/f'icon_{i:02d}.svg'
        report=raster_to_svg(tile,dest)
        info.append(report)
    return info


if __name__=='__main__':
    p=argparse.ArgumentParser();p.add_argument('image');p.add_argument('--out',default='editable_icon.svg')
    p.add_argument('--colors',type=int,default=4)
    a=p.parse_args()
    print(raster_to_svg(Image.open(a.image),a.out,colors=a.colors))


In [ ]:
%%writefile iconflow/evaluate_flow.py
"""Held-out conditional-vector-field MSE (diagnostic, not a visual quality benchmark)."""
import argparse,json
from pathlib import Path
import torch
from torch.nn import functional as F
from iconflow.train import VelocityUNet,make_icon,flow_interpolant
import numpy as np

def evaluate(checkpoint,n=80,seed=454,size=None,batch=16):
    torch.manual_seed(seed)
    device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    saved=torch.load(checkpoint,map_location=device,weights_only=True)
    size=int(size or saved['size'])
    model=VelocityUNet(base=int(saved['width'])).to(device)
    model.load_state_dict(saved['model']);model.eval()
    tot,baseline=0.,0.
    with torch.no_grad():
        for first in range(0,n,batch):
            images=[];classes=[]
            for j in range(first,min(first+batch,n)):
                # indices far beyond 0..train_samples to avoid exact training-image duplicates
                im,cls=make_icon(1_000_000+j,size=size,seed=seed+10)
                images.append(torch.tensor(np.asarray(im,dtype=np.float32).copy()).permute(2,0,1)/127.5-1)
                classes.append(cls)
            x1=torch.stack(images).to(device); c=torch.tensor(classes,device=device)
            x0=torch.randn_like(x1);t=torch.rand(len(c),device=device)
            xt,target=flow_interpolant(x0,x1,t,sigma_min=float(saved.get('sigma_min',.01)))
            pred=model(xt,t,c)
            tot+=float(F.mse_loss(pred,target,reduction='sum').item())
            baseline+=float(target.square().sum().item())
    denom=n*3*size*size
    return {'held_out_conditional_mse':tot/denom,'zero_velocity_mse':baseline/denom,
            'held_out_n':n,'training_sample_overlap':'No index overlap; same procedural generator family',
            'caveat':'Not likelihood, FID, recognition score, or a paper benchmark'}

if __name__=='__main__':
    p=argparse.ArgumentParser();p.add_argument('--checkpoint',required=True)
    p.add_argument('--n',type=int,default=80);p.add_argument('--seed',type=int,default=454)
    p.add_argument('--out',default='flow_eval.json');a=p.parse_args()
    if a.n<1:raise ValueError('n must be positive')
    result=evaluate(a.checkpoint,n=a.n,seed=a.seed)
    Path(a.out).write_text(json.dumps(result,indent=2))
    print(json.dumps(result,indent=2))


In [ ]:
%%writefile iconflow/demo.py
"""Class-conditioned creative asset generator + explicit classical SVG trace postprocess."""
import argparse
import tempfile
from pathlib import Path
import torch
from PIL import Image
from iconflow.train import VelocityUNet,CLASSES,sample,to_pil
from iconflow.vectorize import raster_to_svg

PALETTES={'Original (sampled)':None,'Cobalt':'#4763D8','Coral':'#E95A67','Jade':'#269D87','Ink':'#152642'}


def generate_one(model,device,size,kind,seed,steps,palette,solver="euler",guidance=1.):
    with torch.random.fork_rng(devices=[torch.cuda.current_device()] if device.type=='cuda' else []):
        torch.manual_seed(int(seed))
        if device.type=='cuda':torch.cuda.manual_seed_all(int(seed))
        labels=torch.tensor([CLASSES.index(kind)],device=device)
        img=to_pil(sample(model,labels,size=size,steps=int(steps),device=device,solver=solver,guidance_scale=float(guidance))[0])
    dest=Path(tempfile.mkdtemp(prefix='icon_assets_'))
    img.resize((384,384),Image.Resampling.NEAREST).save(dest/'icon.png')
    report=raster_to_svg(img,dest/'editable_icon.svg',foreground=PALETTES[palette])
    return Image.open(dest/'icon.png'),str(dest/'editable_icon.svg'),f'{report["paths"]} vector contours (heuristic; inspect and refine in Illustrator)'


def create_app(checkpoint='outputs/flow_model.pt'):
    import gradio as gr
    saved=torch.load(checkpoint,map_location='cpu',weights_only=True)
    device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model=VelocityUNet(base=saved.get('width',32)).to(device).eval()
    model.load_state_dict(saved['model'])
    cfg_trained=bool(saved.get('cfg_dropout',0)>0)
    def predict(kind,seed,steps,palette,solver,guidance):
        if float(guidance)!=1. and not cfg_trained:
            raise gr.Error('Train with --cfg-dropout >0 before using classifier-free guidance')
        return generate_one(model,device,saved.get('size',24),kind,seed,steps,palette,solver,guidance)
    return gr.Interface(predict,
        inputs=[gr.Dropdown(CLASSES,value='star',label='Asset class'),
                gr.Slider(0,9999,step=1,value=42,label='Seed'),
                gr.Slider(10,100,step=5,value=35,label='ODE steps'),
                gr.Dropdown(list(PALETTES),value='Cobalt',label='Vector palette'),
                gr.Dropdown(['euler','heun'],value='heun',label='ODE sampler'),
                gr.Slider(0,3,step=.25,value=1,label='Classifier-free guidance scale (needs CFG-trained checkpoint)')],
        outputs=[gr.Image(label='Generated raster'),gr.File(label='Editable SVG'),gr.Textbox(label='Trace diagnostics')],
        title='IconFlow · Generative Icons + Editable Vector Export',
        description='From-scratch conditional Flow Matching + optional CFG + ODE solver. SVG uses classical contour tracing. A checkpoint trained without label dropout only supports scale=1. Quality depends on training.')

if __name__=='__main__':
    p=argparse.ArgumentParser();p.add_argument('--checkpoint',default='outputs/flow_model.pt')
    create_app(p.parse_args().checkpoint).launch(share=False)


In [ ]:
%%writefile pytest.ini
[pytest]
testpaths = tests
pythonpath = .
addopts = --import-mode=importlib


In [ ]:
%%writefile tests/test_iconflow_advanced.py
import torch
import pytest
from iconflow.train import VelocityUNet,conditional_flow_loss,sample

class ToyVelocity(torch.nn.Module):
    num_classes=5
    def forward(self,x,t,c):
        # The classifier-free null token yields zero velocity;
        # all other labels yield 1: CFG with scale=2 must double it.
        return (c!=self.num_classes).float()[:,None,None,None].expand_as(x)

def test_classifier_free_guidance_math():
    m=ToyVelocity();labels=torch.tensor([0]);noise=torch.zeros(1,3,8,8)
    baseline=sample(m,labels,size=8,steps=3,noise=noise,clip_output=False,guidance_scale=1)
    double=sample(m,labels,size=8,steps=3,noise=noise,clip_output=False,guidance_scale=2)
    assert torch.allclose(baseline,torch.ones_like(baseline),atol=1e-6)
    assert torch.allclose(double,torch.full_like(double,2),atol=1e-6)

def test_cfg_null_dropout_trains_unconditional_embeddings():
    torch.manual_seed(11)
    net=VelocityUNet(base=8)
    x=torch.randn(2,3,16,16);labels=torch.tensor([0,1],dtype=torch.long)
    loss=conditional_flow_loss(net,x,labels,cfg_dropout=1.)
    # Output head is intentionally initialized to zero, so embeddings get no
    # gradient on step one. After one optimizer step, CFG null-token gradients flow.
    opt=torch.optim.Adam(net.parameters(),lr=1e-3)
    loss.backward();assert torch.isfinite(loss)
    opt.step();opt.zero_grad()
    loss=conditional_flow_loss(net,x,labels,cfg_dropout=1.)
    loss.backward()
    assert net.label.weight.grad[-1].abs().sum()>0
    assert net.label.weight.grad[:-1].abs().sum()==0

def test_invalid_cfg_rejected():
    net=VelocityUNet(base=8)
    with pytest.raises(ValueError):
        conditional_flow_loss(net,torch.zeros(1,3,16,16),torch.zeros(1,dtype=torch.long),cfg_dropout=1.1)
    with pytest.raises(ValueError):
        sample(net,torch.tensor([1]),size=16,steps=1,guidance_scale=-1)


In [ ]:
%%writefile tests/test_iconflow_paper_fidelity.py
"""Mathematical checks of conditional OT path, velocity, and ODE solvers."""
import math
import torch
from torch import nn
from iconflow.train import flow_interpolant, sample, VelocityUNet, conditional_flow_loss


def test_gaussian_conditional_path_endpoints_velocity():
    a=torch.zeros(3,3,8,8);b=torch.ones_like(a)
    a[1]=2;a[2]=-.5
    t=torch.tensor([0.,.5,1.])
    sigma=.01
    xt,v=flow_interpolant(a,b,t,sigma)
    assert torch.allclose(xt[0],a[0])
    assert torch.allclose(xt[2],b[2]+sigma*a[2])
    assert torch.allclose(v,b-(1-sigma)*a)
    try:flow_interpolant(a,b,t,1.01)
    except ValueError:pass
    else:raise AssertionError('invalid sigma_min accepted')


def test_flow_loss_has_finite_gradients():
    model=VelocityUNet(base=8)
    x=torch.randn(2,3,16,16)
    y=torch.tensor([0,4])
    loss=conditional_flow_loss(model,x,y,sigma_min=.01,
                               noise=torch.zeros_like(x),times=torch.tensor([0.,1.]))
    loss.backward()
    assert torch.isfinite(loss)
    assert model.inconv.weight.grad is not None


class LinearVelocity(nn.Module):
    def forward(self,x,t,c):
        return -x


def test_heun_converges_better_than_euler_for_linear_ode():
    model=LinearVelocity();labels=torch.tensor([1]);noise=torch.full((1,3,8,8),.5)
    true=.5/math.e
    e=sample(model,labels,size=8,steps=4,noise=noise,solver='euler',clip_output=False)
    h=sample(model,labels,size=8,steps=4,noise=noise,solver='heun',clip_output=False)
    assert abs(h.mean().item()-true)<abs(e.mean().item()-true)


def test_sampler_accepts_cpu_labels_and_validates_noise_shape():
    class Zeros(nn.Module):
        def forward(self,x,t,c): return torch.zeros_like(x)
    labels=torch.tensor([0,2]); noise=torch.randn(2,3,8,8)
    out=sample(Zeros(),labels,size=8,steps=5,solver='euler',noise=noise,clip_output=False)
    assert torch.allclose(out,noise)
    try: sample(Zeros(),labels,size=8,steps=1,noise=noise[:1])
    except ValueError: pass
    else: raise AssertionError('Should reject mismatched noise shapes')


def test_exact_constant_velocity_integration_euler_heun():
    class Constant(nn.Module):
        def forward(self,x,t,c): return torch.ones_like(x)*2
    labels=torch.tensor([0]); noise=torch.zeros(1,3,8,8)
    for solver in ('euler','heun'):
        out=sample(Constant(),labels,size=8,steps=3,solver=solver,noise=noise,clip_output=False)
        assert torch.allclose(out,torch.full_like(noise,2.0),atol=1e-6)


In [ ]:
%%writefile tests/test_iconflow_project.py
import torch
from iconflow.train import VelocityUNet,Icons,conditional_flow_loss,sample

def test_flow_training_and_sampling():
    model=VelocityUNet(base=8)
    ds=Icons(size=16,count=6)
    x=torch.stack([ds[i][0] for i in range(2)])
    c=torch.tensor([ds[i][1] for i in range(2)])
    loss=conditional_flow_loss(model,x,c);loss.backward()
    assert torch.isfinite(loss) and model.inconv.weight.grad is not None
    generated=sample(model,labels=c,size=16,steps=2)
    assert generated.shape==(2,3,16,16) and torch.isfinite(generated).all()


def test_editable_vector_paths(tmp_path):
    from iconflow.train import make_icon
    from iconflow.vectorize import raster_to_svg
    import xml.etree.ElementTree as ET
    img,_=make_icon(5,size=48)
    output=tmp_path/'icon.svg'
    report=raster_to_svg(img,output,colors=3,foreground='#4763D8')
    assert report['paths']>0
    root=ET.parse(output).getroot()
    assert root.tag.endswith('svg')
    assert any(e.tag.endswith('path') for e in root.iter())
    assert 'data:image' not in output.read_text()


In [ ]:
!python -m pytest -q tests/test_iconflow_*.py

## Controlled experiments
Each training configuration is a starting point, not evidence of convergence.

In [ ]:
!python -m iconflow.train --steps 900 --size 24 --width 32 --batch-size 32 --cfg-dropout 0 --guidance-scale 1 --solver euler --seed 42 --out baseline

In [ ]:
!python -m iconflow.train --steps 900 --size 24 --width 32 --batch-size 32 --cfg-dropout .15 --guidance-scale 1.5 --solver heun --seed 42 --out extension

In [ ]:
!python -m iconflow.evaluate_flow --checkpoint baseline/flow_model.pt --n 80 --out baseline/eval.json

In [ ]:
!python -m iconflow.evaluate_flow --checkpoint extension/flow_model.pt --n 80 --out extension/eval.json

## Preserve artifacts
Download before the Colab runtime is recycled.

In [ ]:
from pathlib import Path
from google.colab import files
import shutil
for folder in ('baseline','extension','paired_eval','pipeline_smoke_reward_probe','learned_proxy_smoke','eval_baseline','eval_extension'):
    if Path(folder).is_dir():
        archive=shutil.make_archive(folder,'zip',folder)
        files.download(archive)